# Setup
This notebook is for training `classification model` on the latents of `ImageNet` so that we can get its feature layers as backbone for our `Unet` (At `DownSampling`)

In [ ]:
# Kaggle Setup
!git clone https://github.com/fastai/course22p2.git
!pip install -q numpy --upgrade          # upgrade numpy FIRST
!pip install -q fastcore einops datasets timm k-diffusion torcheval

In [ ]:
# Kaggle Setup
import sys
sys.path.append('/kaggle/working/course22p2')
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

In [ ]:
import pickle,gzip

from glob import glob
from torcheval.metrics import MulticlassAccuracy

from miniai.imports import *

from fastprogress import progress_bar
from diffusers import AutoencoderKL

torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)
mpl.rcParams['figure.dpi'] = 70

if fc.defaults.cpus>8: fc.defaults.cpus=8

In [ ]:
path_data = Path('data')/'ILSVRC'
path = path_data/'Data'/'CLS-LOC'

dest = path_data/'latents'
dest.mkdir(exist_ok=True)

# Data

In [ ]:
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-ema").cuda().requires_grad_(False)

In [ ]:
class ImageDS:
  def __init__(self, path, spec):
    cache = path/'files.zpkl'
    if cache.exists():
      with gzip.open(cache) as f: self.files = pickle.load(f)  # already cached → load instantly
    else:
      self.files = glob(str(path/spec), recursive=True)        # first run → slow disk scan
      with gzip.open(cache, 'wb', compresslevel=1) as f: pickle.dump(self.files, f)    # save for next time
 
  def __len__(self): return len(self.files)
  def __getitem__(self, i):
    f = self.files[i]
    im = read_image(f, mode=ImageReadMode.RGB)/255
    im = TF.resize(TF.center_crop(im, min(im.shape[1:])), 256) # center_crop to square: e.g. 480x640 → 480x480 (center of image kept)
    return im,f

- `compresslevel=1` = fastest compression, not smallest
- `VAE` expects square input,

    WHY center_crop BEFORE resize:
     images have different aspect ratios: 480x640, 1920x1080, 300x200, etc.
     if you resize directly to 256x256 → stretches/squishes the image
     so: crop to square first (preserve aspect ratio) → then resize to 256x256
    
     480x640 → center_crop(480) → 480x480 → resize → 256x256 ✓
     480x640 → resize directly → 256x256 ✗ (squished horizontally)
    
    - 'wb' = write binary
    - 'w'  = write text  (strings only)
    - 'b'  = binary mode (bytes, required for pickle and gzip)

In [ ]:
bs = 64
ds = ImageDS(path, '**/*.JPEG')
dl = DataLoader(ds, batch_size=bs, num_workers=4)

In [ ]:
xb,yb = next(iter(dl))
xs = vae.encode(xb.cuda()).latent_dist.mean
xs.shape

In [ ]:
show_images(((xs[:16,:3])/4).sigmoid(), imsize=2)

In [ ]:
xd = to_cpu(vae.decode(xs))
show_images(xd['sample'][:16].clamp(0,1), imsize=2)

In [ ]:
# Same result as the npmm file we made before
if not dest.exist():
  dest.mkdir()
  for xb, yb in progress_bar(dl):
    eb = to_cpu(vae.encode(xb.cuda()).latent_dist.mean).numpy()
    for ebi, ybi ub zip(eb,yb):
      ybi = dest/path(ybi).relative_to(path).with_suffix('')
      (ybi.parent).mkdir(parents=True, exist_ok=True)
      np.save(ybi, ebi)

    for ebi, ybi in zip(eb, yb):          # loop over each image in batch
            ybi = dest/Path(ybi)          # build save path mirroring original structure
                  .relative_to(path)         # strip the source root
                  .with_suffix('')           # remove .JPEG → will become .npy
            
            ybi.parent.mkdir(...)             # create subfolder if needed
            np.save(ybi, ebi)                 # save latent as .npy file

In [ ]:
class NumpyDS(ImageDS):
  def __getitem__(self, i):
    f = self.files[i]
    im = np.load(f)
    return im, f

In [ ]:
bs = 128

In [ ]:
tds = NumpyDS(dest/'train', '**/*.npy')
vds = NumpyDS(dest/'val', '**/*.npy')

In [ ]:
tdl = DataLoader(tds, batch_size=bs, num_workers=0)
xb,yb = next(iter(tdl))
xb.mean((0,2,3)),xb.std((0,2,3))  # mean,std -> for each filter(channel)

In [ ]:
xmean,xstd = (tensor([ 5.37007,  2.65468,  0.44876, -2.39154]),
              tensor([3.99512, 4.44317, 3.21629, 3.10339]))

In [ ]:
class TfmDS:
  def __init__(self, ds ,tfmx=fc.noop, tfmy=fc.noop): fc.store_attr()
  def __len__(self): return len(self.ds)
  def __getitem__(self, i):
    x,y = self.ds[i]
    return self.tfmx(x), self.tfmy(y)

In [ ]:
id2str = (path_data/'imagenet_lsvrc_2015_synsets.txt').read_text().splitlines()
str2id = {j:i for i,j in enumerate(id2str)}

In [ ]:
aug_tfms = nn.Sequential(T.Pad(2), T.RandomCrop(32), RandErase())
norm_tfm = T.Normalize(xmean, xstd)

In [ ]:
def tfmx(x, aug=False):
    x = norm_tfm(tensor(x))
    if aug: x = aug_tfms(x[None])[0]
    return x

def tfmy(y): return tensor(str2id[Path(y).parent.name])

tfm_tds = TfmDS(tds, partial(tfmx, aug=True), tfmy)
tfm_vds = TfmDS(vds, tfmx, tfmy)

In [ ]:
def denorm(x): return (x*xstd[:,None,None]+xmean[:,None,None])

In [ ]:
dls = DataLoaders(*get_dls(tfm_tds, ttfm_vds, bs=bs, num_workers=4))

In [ ]:
all_synsets = [o.split('\t') for o in (path_data/'words.txt').read_text().splitlines()]
synsets = {k:v.split(',', maxsplit=1)[0] for k,v in all_synsets if k in id2str}

that is just for display not training

In [ ]:
xb,yb = next(iter(dls.train))
titles = [synsets[id2str[o]] for o in yb]
xb.mean(), xb.std()

In [ ]:
xd = to_cpu(vae.decode(denorm(xb[:9]).cuda()))
show_images(xd['sample'].clamp(0,1), imsize=4, titles=titles[:9])

# Training

In [ ]:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)

opt_func = partial(optim.AdamW, eps=1e-5)
metrics = MetricsCB(accuracy=MulticlassAccuracy())
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), MixedPrecision()]

In [ ]:
def conv(ni, nf, ks=3, stride=1, act=nn.ReLU, norm=None, bias=True):
    layers = []
    if norm: layers.append(norm(ni))
    if act : layers.append(act())
    layers.append(nn.Conv2d(ni, nf, stride=stride, kernel_size=ks, padding=ks//2, bias=bias))
    return nn.Sequential(*layers)

def _conv_block(ni, nf, stride, act=act_gr, norm=None, ks=3):
    return nn.Sequential(conv(ni, nf, stride=1     , act=act, norm=norm, ks=ks),
                         conv(nf, nf, stride=stride, act=act, norm=norm, ks=ks))

class ResBlock(nn.Module):
    def __init__(self, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
        super().__init__()
        self.convs = _conv_block(ni, nf, stride, act=act, ks=ks, norm=norm)
        self.idconv = fc.noop if ni==nf else conv(ni, nf, ks=1, stride=1, act=None, norm=norm)
        self.pool = fc.noop if stride==1 else nn.AvgPool2d(2, ceil_mode=True)

    def forward(self, x): return self.convs(x) + self.idconv(self.pool(x))

def res_blocks(n_bk, ni, nf, stride=1, ks=3, act=act_gr, norm=None):
    return nn.Sequential(*[
        ResBlock(ni if i==0 else nf, nf, stride=stride if i==n_bk-1 else 1, ks=ks, act=act, norm=norm)
        for i in range(n_bk)])

def get_dropmodel(nfs, nbks, act=act_gr, norm=nn.BatchNorm2d, drop=0.2):
    layers = [nn.Conv2d(4, nfs[0], 5, padding=2)]
    layers += [res_blocks(nbks[i], nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
               for i in range(len(nfs)-1)]
    layers += [act_gr(), norm(nfs[-1]), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(drop)]
    layers += [nn.Linear(nfs[-1], 1000, bias=False), nn.BatchNorm1d(1000)]
    return nn.Sequential(*layers).apply(iw)

copy-pasted from previous notebooks nothing changed

In [ ]:
epochs = 40
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
model = get_dropmodel(nbks=(1,2,4,3), nfs=(32, 64, 128, 512, 1024), drop=0.1)
learn = Learner(model, dls, F.cross_entropy, lr=lr, cbs=cbs+xtra, opt_func=opt_func)

    WHY More Deeper at the end:
      Early layers = simple edges, colors → don't need much capacity
      Deep layers = semantic content, objects → need many channels + blocks
      Wasting blocks at early layers = expensive with no benefit

    - (1,2,4,3) -> lowered to 3 is because this might be more for classification
                   than for features backbone for our Unet

    WHERE IT CAME FROM:
      "Simple Diffusion" paper — showed that standard U-Net wastes compute
      by having equal capacity at every resolution.
      Most of the information loss/gain happens at the bottleneck, not the edges.


In [ ]:
learn.fit(epochs)

In [ ]:
torch.save(learn.model,'/kaggle/working/ImageNet_Latents_Classification.pkl')